In [ ]:
import random
import json

class SupplyChainWorld:
    """
    The Global Simulation Engine - India Edition.
    Generates supply chain states and crises for the LLM Agent to solve.
    """
    def __init__(self):
        # Indian Logistics Hubs and their starting inventory
        self.warehouses = {
            "Mumbai": {"inventory": 100, "capacity": 200, "status": "Operational"},
            "Delhi": {"inventory": 150, "capacity": 200, "status": "Operational"},
            "Bangalore": {"inventory": 80, "capacity": 200, "status": "Operational"},
            "Chennai": {"inventory": 120, "capacity": 200, "status": "Operational"}
        }

        # Transit costs in Indian Rupees (₹)
        self.transit_costs = {
            "Bangalore_to_Delhi": 12000,
            "Delhi_to_Mumbai": 8000,
            "Chennai_to_Mumbai": 10000,
            "Bangalore_to_Chennai": 5000
        }

    def generate_crisis(self):
        """Generates a random supply chain crisis."""
        crises = [
            {
                "event": "Monsoon Flooding",
                "location": "Mumbai",
                "impact": "Major flooding in warehouses. 40% inventory damaged. Transit slowed.",
                "urgency": "High"
            },
            {
                "event": "Festival Demand Spike",
                "location": "Delhi",
                "impact": "Diwali demand spike. Orders increased by 400%. Stockout imminent.",
                "urgency": "Critical"
            },
            {
                "event": "Transport Strike",
                "location": "Bangalore",
                "impact": "Local trucking strike. Outbound shipments halted for 5 days.",
                "urgency": "Medium"
            }
        ]
        return random.choice(crises)

    def get_current_state(self):
        """Returns the full state of the world as a JSON string."""
        full_state = {
            "warehouses": self.warehouses,
            "transit_costs": self.transit_costs,
            "currency": "INR (₹)"
        }
        return json.dumps(full_state, indent=2)


In [ ]:
!pip install -q -U google-generativeai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.1/155.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.5/320.5 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-dataproc 5.32.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-firestore 2.34.1 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-storage-control 1.16.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-language 2.22.0 requires protobuf<8.0.0,>=6.33.5, but you have

In [ ]:
class SupplyChainAgent:
    """
    The LLM-powered Strategic Manager.
    Uses Chain-of-Thought reasoning to solve Indian supply chain crises.
    """
    def __init__(self, world):
        self.world = world

    def create_prompt(self, crisis, state):
        prompt = f"""
        You are the Strategic Logistics Manager for a major Indian supply chain.

        GOAL: Resolve the crisis while minimizing cost (in Indian Rupees ₹) and maximizing speed.

        CURRENT STATE:
        {state}

        CURRENT CRISIS:
        Event: {crisis['event']}
        Location: {crisis['location']}
        Impact: {crisis['impact']}
        Urgency: {crisis['urgency']}

        You must respond in the following JSON format:
        {{
            "reasoning": "Step-by-step logic on why this is the best move",
            "action": "The specific action to take (e.g., Move stock from Bangalore to Delhi)",
            "estimated_cost": "The cost in ₹ calculated from the transit_costs",
            "expected_impact": "How this solves the crisis"
        }}

        Return ONLY the JSON.
        """
        return prompt

    def solve_crisis(self, crisis):
        state = self.world.get_current_state()
        prompt = self.create_prompt(crisis, state)
        response = model.generate_content(prompt)
        text_response = response.text.replace('```json', '').replace('```', '').strip()
        try:
            return json.loads(text_response)
        except json.JSONDecodeError:
            return {"error": "LLM failed to output valid JSON", "raw": text_response}


In [ ]:
import google.generativeai as genai
import json

# 1. Setup the LLM
GOOGLE_API_KEY = "AQ.Ab8RN6KE49A1OmW2uDI64eFfekXViVuh65y6VVUssasv_XtM_A" # <--- PASTE YOUR KEY HERE
genai.configure(api_key=GOOGLE_API_KEY)

def find_working_model():
    """Queries the API to find the best available model for this account."""
    print("Searching for available models...")
    try:
        available_models = [m.name for m in genai.list_models()]

        # UPDATED: Priority list matching your specific available models
        priority_list = [
            'models/gemini-3.8-flash',
            'models/gemini-2.5-flash',
            'models/gemini-flash-latest',
            'models/gemini-pro-latest'
        ]

        for target in priority_list:
            if target in available_models:
                print(f"✅ Found working model: {target}")
                return genai.GenerativeModel(target)

        # Final fallback: just take the first model that contains 'flash' or 'pro'
        for m in available_models:
            if 'flash' in m or 'pro' in m:
                print(f"✅ Using fallback model: {m}")
                return genai.GenerativeModel(m)
    except Exception as e:
        print(f"❌ Error listing models: {e}")

    return None

# Initialize the model automatically
model = find_working_model()

if model is None:
    raise Exception("Could not find a working Gemini model. Please check your API key.")

class SupplyChainAgent:
    """
    The LLM-powered Strategic Manager.
    Uses Chain-of-Thought reasoning to solve supply chain crises.
    """
    def __init__(self, world):
        self.world = world

    def create_prompt(self, crisis, state):
        prompt = f"""
        You are the Strategic Logistics Manager of a global supply chain.

        GOAL: Resolve the crisis while minimizing cost and maximizing speed.

        CURRENT STATE:
        {state}

        CURRENT CRISIS:
        Event: {crisis['event']}
        Location: {crisis['location']}
        Impact: {crisis['impact']}
        Urgency: {crisis['urgency']}

        You must respond in the following JSON format:
        {{
            "reasoning": "Step-by-step logic on why this is the best move",
            "action": "The specific action to take (e.g., Move stock from Chicago to New York)",
            "estimated_cost": "The cost calculated from the transit_costs",
            "expected_impact": "How this solves the crisis"
        }}

        Return ONLY the JSON.
        """
        return prompt

    def solve_crisis(self, crisis):
        state = self.world.get_current_state()
        prompt = self.create_prompt(crisis, state)

        try:
            response = model.generate_content(prompt)
            text_response = response.text.replace('```json', '').replace('```', '').strip()
            return json.loads(text_response)
        except Exception as e:
            return {"error": str(e), "raw": "Failed to generate or parse response"}

# --- TEST THE AGENT ---
# Initialize the world and the agent
# Make sure the SupplyChainWorld class is already defined in your notebook!
world = SupplyChainWorld()
agent = SupplyChainAgent(world)

# Generate a crisis
crisis = world.generate_crisis()
print(f"\n🚩 CRISIS: {crisis['event']} in {crisis['location']}")
print("-" * 30)

# Let the agent solve it
decision = agent.solve_crisis(crisis)

if "error" in decision:
    print("❌ Error:", decision["error"])
else:
    print(f"🧠 REASONING: {decision['reasoning']}")
    print(f"🚀 ACTION: {decision['action']}")
    print(f"💰 COST: {decision['estimated_cost']}")
    print(f"🎯 IMPACT: {decision['expected_impact']}")


Searching for available models...
✅ Found working model: models/gemini-3.8-flash

🚩 CRISIS: Warehouse Flood in Houston
------------------------------
🧠 REASONING: Houston lost 50% of its inventory (60 units), dropping its stock to 60. To maintain fulfillment capabilities and regional coverage while respecting network topology, Houston requires replenishment. The only available inbound route to Houston in the network is from Los Angeles (cost: 600). Transferring a replenishment batch from Los Angeles (which currently holds 80 units) to Houston stabilizes inventory levels at the lowest available routing cost without exceeding Houston's reduced capacity.
🚀 ACTION: Move stock from Los Angeles to Houston
💰 COST: $600
🎯 IMPACT: Restores critical inventory levels at the Houston facility, compensating for flood losses and stabilizing fulfillment operations in the region.


In [ ]:
class RewardModel:
    """
    The Critic.
    Evaluates the LLM's decisions and provides a numerical reward in INR context.
    """
    def __init__(self, world):
        self.world = world

    def evaluate_action(self, crisis, decision):
        if "error" in decision:
            return -100, "Invalid Action"

        reward = 0
        feedback = []

        # 1. Effectiveness
        if crisis['location'] in decision['action']:
            reward += 100
            feedback.append("✅ Crisis location addressed.")
        else:
            reward -= 100
            feedback.append("❌ Failed to address the crisis location.")

        # 2. Cost Efficiency (Using INR logic)
        try:
            # Remove ₹, commas, and spaces from the cost string
            cost_str = decision['estimated_cost'].replace('₹', '').replace(',', '').replace(' ', '').strip()
            cost = float(''.join(filter(str.isdigit, cost_str)))

            if cost <= 15000: # Threshold for a "reasonable" cost in INR
                reward += 50
                feedback.append("✅ Cost-effective routing.")
            else:
                reward -= 50
                feedback.append("❌ Route is too expensive.")
        except:
            reward -= 20
            feedback.append("⚠️ Could not verify cost.")

        # 3. Risk Check
        for city, data in self.world.warehouses.items():
            if city in decision['action'] and data['inventory'] < 100:
                reward -= 100
                feedback.append(f"❌ Risk: Took stock from {city} which is already low!")

        return reward, " | ".join(feedback)


In [ ]:
class FlawedRewardModel:
    """
    A flawed reward model that only cares about cost.
    This is designed to demonstrate 'Reward Hacking'.
    """
    def __init__(self, world):
        self.world = world

    def evaluate_action(self, crisis, decision):
        if "error" in decision:
            return -100, "Invalid Action"

        reward = 0
        feedback = []

        # FLAW: We give a huge reward for LOW COST, but a small reward for effectiveness.
        # The AI will learn that 'Doing Nothing' or 'Cheap Moves' is better than 'Solving the Crisis'.

        try:
            cost_str = decision['estimated_cost'].replace('₹', '').replace(',', '').replace(' ', '').strip()
            cost = float(''.join(filter(str.isdigit, cost_str)))

            if cost == 0: # The agent discovers that doing nothing costs 0!
                reward += 200
                feedback.append("✅ Super Low Cost (Reward Hacked!)")
            elif cost < 5000:
                reward += 100
                feedback.append("✅ Low Cost.")
            else:
                reward -= 50
                feedback.append("❌ Too Expensive.")
        except:
            reward -= 20

        # The effectiveness reward is too low to matter
        if crisis['location'] in decision['action']:
            reward += 10
            feedback.append("✅ Addressed location.")

        return reward, " | ".join(feedback)

# --- TEST THE FLAWED SYSTEM ---
flawed_reward = FlawedRewardModel(world)
crisis = world.generate_crisis()
decision = agent.solve_crisis(crisis)
score, feedback = flawed_reward.evaluate_action(crisis, decision)

print(f"🚩 CRISIS: {crisis['event']}")
print(f"🚀 ACTION: {decision['action']}")
print(f"📈 REWARD: {score} | Feedback: {feedback}")


🚩 CRISIS: Warehouse Flood
🚀 ACTION: Transfer 40 units of stock from Los Angeles to Houston
📈 REWARD: 110 | Feedback: ✅ Low Cost. | ✅ Addressed location.


In [ ]:
class HardenedRewardModel:
    """
    The Professional Reward Model.
    Prevents reward hacking by penalizing inactivity and prioritizing effectiveness.
    """
    def __init__(self, world):
        self.world = world

    def evaluate_action(self, crisis, decision):
        if "error" in decision:
            return -100, "Invalid Action"

        reward = 0
        feedback = []

        # 1. MANDATORY EFFECTIVENESS: If you don't solve it, you get a massive penalty.
        if crisis['location'] in decision['action']:
            reward += 200 # High reward for solving
            feedback.append("✅ Crisis Solved.")
        else:
            reward -= 300 # MASSIVE penalty for ignoring the crisis (Stops reward hacking)
            feedback.append("❌ Crisis Ignored!")

        # 2. COST EFFICIENCY: Now that effectiveness is guaranteed, we optimize for cost.
        try:
            cost_str = decision['estimated_cost'].replace('₹', '').replace(',', '').replace(' ', '').strip()
            cost = float(''.join(filter(str.isdigit, cost_str)))
            if cost <= 15000:
                reward += 50
                feedback.append("✅ Efficient Cost.")
            else:
                reward -= 50
                feedback.append("❌ Over Budget.")
        except:
            reward -= 20

        return reward, " | ".join(feedback)

# --- TEST THE HARDENED SYSTEM ---
hardened_reward = HardenedRewardModel(world)
decision = agent.solve_crisis(crisis)
score, feedback = hardened_reward.evaluate_action(crisis, decision)

print(f"🚩 CRISIS: {crisis['event']}")
print(f"🚀 ACTION: {decision['action']}")
print(f"📈 REWARD: {score} | Feedback: {feedback}")


🚩 CRISIS: Warehouse Flood
🚀 ACTION: Transfer 50 units of inventory from Los Angeles to Houston
📈 REWARD: 250 | Feedback: ✅ Crisis Solved. | ✅ Efficient Cost.


In [ ]:
# This code creates a file called requirements.txt in your Colab environment
requirements_content = """
google-generativeai
numpy
matplotlib
json
"""

with open('requirements.txt', 'w') as f:
    f.write(requirements_content.strip())

print("✅ requirements.txt has been created successfully!")


✅ requirements.txt has been created successfully!


In [ ]:
!pip install -q streamlit google-generativeai
!npm install -g localtunnel


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 95.8 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
added 22 packages in 2s
⠙
⠙3 packages are looking for funding
⠙  run `npm fund` for details
⠙

In [ ]:
%%writefile app.py
import streamlit as st
import google.generativeai as genai
import json
import random

# ==========================================
# 1. THE WORLD ENGINE
# ==========================================
class SupplyChainWorld:
    def __init__(self):
        self.warehouses = {
            "Mumbai": {"inventory": 100, "capacity": 200, "status": "Operational"},
            "Delhi": {"inventory": 150, "capacity": 200, "status": "Operational"},
            "Bangalore": {"inventory": 80, "capacity": 200, "status": "Operational"},
            "Chennai": {"inventory": 120, "capacity": 200, "status": "Operational"}
        }
        self.transit_costs = {
            "Bangalore_to_Delhi": 12000,
            "Delhi_to_Mumbai": 8000,
            "Chennai_to_Mumbai": 10000,
            "Bangalore_to_Chennai": 5000
        }

    def generate_crisis(self):
        crises = [
            {"event": "Monsoon Flooding", "location": "Mumbai", "impact": "Major flooding. 40% inventory damaged.", "urgency": "High"},
            {"event": "Festival Demand Spike", "location": "Delhi", "impact": "Orders increased by 400%. Stockout imminent.", "urgency": "Critical"},
            {"event": "Transport Strike", "location": "Bangalore", "impact": "Trucking strike. Outbound shipments halted.", "urgency": "Medium"}
        ]
        return random.choice(crises)

    def get_current_state(self):
        return json.dumps({"warehouses": self.warehouses, "transit_costs": self.transit_costs, "currency": "INR (₹)"}, indent=2)

# ==========================================
# 2. THE AI AGENT
# ==========================================
class SupplyChainAgent:
    def __init__(self, api_key, world):
        genai.configure(api_key=api_key)
        # FIXED: Updated to your working model gemini-3.8-flash
        self.model = genai.GenerativeModel('gemini-3.8-flash')
        self.world = world

    def solve_crisis(self, crisis):
        state = self.world.get_current_state()
        prompt = f"""
        You are the Strategic Logistics Manager of a major Indian supply chain.
        GOAL: Resolve the crisis while minimizing cost (INR ₹) and maximizing speed.
        CURRENT STATE: {state}
        CURRENT CRISIS: {crisis}
        Respond ONLY in JSON: {{"reasoning": "...", "action": "...", "estimated_cost": "...", "expected_impact": "..."}}
        """
        try:
            response = self.model.generate_content(prompt)
            text = response.text.replace('```json', '').replace('```', '').strip()
            return json.loads(text)
        except Exception as e:
            return {"error": str(e)}

# ==========================================
# 3. THE REWARD CRITIC
# ==========================================
class RewardModel:
    def __init__(self, world):
        self.world = world

    def evaluate(self, crisis, decision):
        if "error" in decision: return -100, "Invalid Action"
        reward = 0
        feedback = []
        if crisis['location'] in decision['action']:
            reward += 100
            feedback.append("✅ Crisis location addressed.")
        else:
            reward -= 100
            feedback.append("❌ Failed to address location.")

        try:
            cost_str = decision['estimated_cost'].replace('₹', '').replace(',', '').strip()
            cost = float(''.join(filter(str.isdigit, cost_str)))
            if cost <= 15000: reward += 50; feedback.append("✅ Cost-effective.")
            else: reward -= 50; feedback.append("❌ Too expensive.")
        except:
            reward -= 20; feedback.append("⚠️ Cost verification failed.")
        return reward, " | ".join(feedback)

# ==========================================
# 4. STREAMLIT UI
# ==========================================
st.set_page_config(page_title="AI Supply Chain Orchestrator", page_icon="🌐")
st.title("🌐 Strategic Supply Chain Orchestrator")
st.markdown("### *LLM-Powered Autonomous Decision System*")

with st.sidebar:
    st.header("Settings")
    api_key = st.text_input("Enter Gemini API Key", type="password")
    st.info("Enter your key to activate the AI Brain.")

if not api_key:
    st.warning("Please enter your API key in the sidebar to start.")
else:
    world = SupplyChainWorld()
    agent = SupplyChainAgent(api_key, world)
    critic = RewardModel(world)

    if st.button("🚀 Generate & Solve Crisis"):
        crisis = world.generate_crisis()

        col1, col2 = st.columns(2)
        with col1:
            st.error(f"**CRISIS DETECTED**\n\n**Event:** {crisis['event']}\n**Location:** {crisis['location']}\n**Urgency:** {crisis['urgency']}")
        with col2:
            st.info(f"**IMPACT**\n\n{crisis['impact']}")

        with st.spinner("AI Agent is reasoning..."):
            decision = agent.solve_crisis(crisis)

        if "error" in decision:
            st.error(f"AI Error: {decision['error']}")
        else:
            st.subheader("🧠 Agent's Strategic Reasoning")
            st.write(decision['reasoning'])

            st.success(f"**Final Action:** {decision['action']}")
            st.write(f"**Estimated Cost:** {decision['estimated_cost']}")
            st.write(f"**Expected Impact:** {decision['expected_impact']}")

            # Reward Section
            score, feedback = critic.evaluate(crisis, decision)
            st.divider()
            st.subheader("📈 Critic's Evaluation")
            st.metric("Reward Score", score)
            st.write(f"**Feedback:** {feedback}")


Overwriting app.py


In [ ]:
import urllib
import socket

# 1. Get your public IP (Needed for localtunnel password)
print("Your Tunnel Password is: ", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

# 2. Start the Streamlit app in the background
!streamlit run app.py & npx localtunnel --port 8501


Your Tunnel Password is:  136.85.227.243
⠙⠹

⠸⠼⠴⠦2026-10-21 07:54:56.235 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.85.227.243:8501

your url is: https://silver-banks-give.loca.lt
/content/app.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [1]:
# Install Cloudflare Tunnel
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb


--2026-10-21 08:08:41--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.10.2/cloudflared-linux-amd64.deb [following]
--2026-10-21 08:08:41--  https://github.com/cloudflare/cloudflared/releases/download/2026.10.2/cloudflared-linux-amd64.deb
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/991bb622-066a-4784-973d-0a2499b971cc?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-10-21T09%3A00%3A30Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64.deb&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4d

In [ ]:
import os

# 1. Start Streamlit in the background
# We use 'nohup' to make sure it keeps running even if the cell finishes
!nohup streamlit run app.py &

# 2. Start the Cloudflare Tunnel
# This will generate a professional .trycloudflare.com link
!cloudflared tunnel --url http://localhost:8501


nohup: appending output to 'nohup.out'
2026-10-21T08:09:19Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-21T08:09:19Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-21T08:09:22Z INF +--------------------------------------------------------------------------------------------+
2026-10-21T08:09:22Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-21T08:09:22Z INF |  https://explor